In [ ]:
import os, glob

print(os.getcwd())
print(os.path.exists("../data/raw"))
print(glob.glob("../data/raw/*.csv")[:3])

In [ ]:
from pathlib import Path
import os
import pandas as pd

root = Path.cwd()
while not (root / "data" / "raw").exists() and root != root.parent:
    root = root.parent
raw_dir = root / "data" / "raw"
print(raw_dir)

frames = []
for path in sorted(raw_dir.glob("[0-9][0-9][0-9][0-9].csv")):
    d = pd.read_csv(path)
    d["file_year"] = int(path.stem)
    frames.append(d)

print(len(frames), "files loaded")
raw_all = pd.concat(frames, ignore_index=True)
print(raw_all.shape)
print(raw_all.groupby("file_year").size())

In [ ]:
reference = list(pd.read_csv(raw_dir / "2024.csv", nrows=0).columns)

for f in frames:
    year = f["file_year"].iloc[0]
    cols_f = [c for c in f.columns if c != "file_year"]
    if cols_f != reference:
        print(year, "| missing:", set(reference) - set(cols_f), "| extra:", set(cols_f) - set(reference), "| same set, different order:", set(cols_f) == set(reference))

print("column check done")

In [ ]:
cols = ["minutes", "w_ace", "w_svpt", "indoor"]
print(raw_all[cols].isna().groupby(raw_all["file_year"]).mean().round(3).to_string())

In [ ]:
print(pd.crosstab(raw_all["file_year"], raw_all["tourney_level"]).to_string())

In [ ]:
cols = ["minutes", "w_ace", "w_svpt", "indoor"]
nd = raw_all[raw_all["tourney_level"] != "D"]
print(nd[cols].isna().groupby(nd["file_year"]).mean().round(3).to_string())

In [ ]:
sub = nd[nd["file_year"].between(2013, 2017)]
print(sub["w_ace"].isna().groupby([sub["file_year"], sub["tourney_level"]]).mean().round(3).unstack().to_string())

In [ ]:
a = raw_all[raw_all["tourney_level"] == "A"]
print(a.groupby("tourney_name")["file_year"].agg(["min", "max", "count"]).to_string())

m = raw_all[(raw_all["tourney_level"] == "M") & (raw_all["file_year"] >= 2015)]
print(pd.crosstab(m["tourney_name"], m["file_year"]).to_string())

In [ ]:
f = raw_all[raw_all["tourney_level"].isin(["M", "F"]) & (raw_all["round"] == "F")]
print(pd.crosstab(f["file_year"], [f["tourney_level"], f["best_of"]]).to_string())